In [ ]:
import matplotlib.pyplot as plt
from keras_facenet import FaceNet
import cv2
import numpy as np
from mtcnn import MTCNN
from sklearn.metrics.pairwise import cosine_similarity

def load_and_detect_face(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Detect faces
    detector = MTCNN()
    result = detector.detect_faces(image_rgb)[0]

    # Crop faces
    x, y, width, height = result['box']
    face = image[y:y+height, x:x+width]

    plt.imshow(cv2.cvtColor(face, cv2.COLOR_BGR2RGB))
    return face


In [21]:
embedder = FaceNet()

def get_face_embedding(image_path):
    img_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

    embeddings = embedder.embeddings([img_rgb])

    return embeddings[0]

In [22]:
# Video face embedding extractor
video_path = 'IMG_7109.mp4'
cap = cv2.VideoCapture(video_path)
detector = MTCNN()

embeddings = []
frame_count = 0
skip_frames = 5

def get_embedding(face_image):
    embeddings = embedder.embeddings([face_image])
    return embeddings[0]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_count % skip_frames == 0:
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = detector.detect_faces(rgb_frame)

        for result in results:
            x, y, w, h = result['box']
            x, y = max(x, 0), max(y, 0)
            face = rgb_frame[y:y+h, x:x+w]

            try:
                embedding = get_embedding(face)
                embeddings.append(embedding)
            except Exception as e:
                print(f"Error embedding face at frame {frame_count}: {e}")

    frame_count += 1

cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 27ms/step


In [23]:
avg_embedding = np.mean(embeddings, axis=0)
print("Average embedding:", avg_embedding)

Average embedding: [-3.82077582e-02  1.68715045e-02 -4.96516898e-02 -2.54676156e-02
  3.21845114e-02  3.66198532e-02 -7.98481982e-03 -4.28947397e-02
  5.95925860e-02 -7.22764153e-03  4.20540087e-02  2.02717070e-04
  1.78427920e-02  9.86701809e-03 -2.10705474e-02 -6.82214368e-03
  6.82000890e-02  4.68208045e-02  2.33289413e-02 -1.96450595e-02
 -2.45151040e-03 -4.40366892e-03  7.64846280e-02 -2.06867885e-02
 -1.95735302e-02 -4.14160490e-02  4.39732708e-02  2.25913934e-02
 -1.71183664e-02 -7.56190100e-04 -3.74462269e-02  1.05620241e-02
 -2.75589712e-02 -2.24108621e-02 -1.70875639e-02  3.37986322e-03
 -8.38328339e-03  3.00871842e-02 -1.53999720e-02 -1.43439164e-02
 -7.24390447e-02 -2.67884731e-02 -7.93250278e-02  2.85434742e-02
  6.52580038e-02 -3.51439342e-02  3.32134329e-02 -1.50253833e-03
 -2.19807941e-02  2.97617856e-02 -2.00173073e-02 -2.62031816e-02
 -2.69760471e-02  8.59070104e-03  7.28782732e-04  7.20622838e-02
 -1.73102412e-02 -1.50921186e-02  2.82954574e-02  2.23006979e-02
  8.32

In [27]:
face1 = load_and_detect_face('Screenshot 2025-04-19 at 16.47.25.png')

# compare with the average embedding
embedding1 = get_face_embedding(face1)

# Example usage (comparing new face with stored face)
score = cosine_similarity(avg_embedding.reshape(1, -1), embedding1.reshape(1, -1))
print(score)

# To get the float value from the 2D output array
similarity_score = score[0][0]
print("Similarity:", similarity_score)


1/1 [==============================] - 0s 27ms/step
[[0.9310031]]
Similarity: 0.9310031
